# AIS Pose Prediction Training

Train the unified pose model from RPY-randomized data. The model predicts port0/port1 `dx, dy` and `dyaw` from image / force-torque / joint inputs. `dz` is handled by the insertion policy.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path('/home/whyz/aic_sejong/ws_aic/src/ais/ais_pose_prediction')
SRC_DIR = Path('/home/whyz/aic_sejong/ws_aic/src')
WS_ROOT = Path('/home/whyz/aic_sejong/ws_aic')

for path in (PROJECT_DIR, SRC_DIR):
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

PROJECT_DIR, WS_ROOT


In [ ]:
import json

import torch
from torch.utils.data import DataLoader

from ais_pose_prediction.model import build_pose_model
from ais_pose_prediction.model.dataset import PosePredictionDataset, load_samples
from ais_pose_prediction.train import _loss, _move_batch, _stats


In [ ]:
DATASET_ROOT = WS_ROOT / 'data' / 'ais_rpy_randomization'
OUTPUT_DIR = WS_ROOT / 'model' / 'ais_pose_prediction' / 'pose_resnet50_v4.0'

VERSIONS = ['v4.0']
EPOCHS = 40
BATCH_SIZE = 16
LR = 1e-4
BACKBONE = 'resnet50'
HIDDEN_DIM = 256
IMAGE_SIZE = 224
NUM_WORKERS = 4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATASET_ROOT, OUTPUT_DIR, DEVICE


In [ ]:
train_samples = load_samples(DATASET_ROOT, versions=VERSIONS, splits=('train',))
val_samples = load_samples(DATASET_ROOT, versions=VERSIONS, splits=('val',))

print(f'train samples: {len(train_samples)}')
print(f'val samples:   {len(val_samples)}')

if not train_samples:
    raise RuntimeError(f'No train samples found under {DATASET_ROOT} for {VERSIONS}')
if not val_samples:
    raise RuntimeError(f'No val samples found under {DATASET_ROOT} for {VERSIONS}')


In [ ]:
stats = _stats(train_samples)
target_mean = {name: stats[name]['mean'] for name in stats}
target_std = {name: stats[name]['std'] for name in stats}

print(json.dumps(stats, indent=2))


In [ ]:
train_ds = PosePredictionDataset(
    train_samples,
    image_size=IMAGE_SIZE,
    target_mean=target_mean,
    target_std=target_std,
)
val_ds = PosePredictionDataset(
    val_samples,
    image_size=IMAGE_SIZE,
    target_mean=target_mean,
    target_std=target_std,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

batch = next(iter(train_loader))
{k: tuple(v.shape) for k, v in batch.items() if torch.is_tensor(v)}


In [ ]:
model = build_pose_model(
    backbone_name=BACKBONE,
    pretrained=True,
    hidden_dim=HIDDEN_DIM,
    num_views=3,
    aggregation='concat',
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

config = {
    'versions': list(VERSIONS),
    'backbone': BACKBONE,
    'hidden_dim': HIDDEN_DIM,
    'image_size': IMAGE_SIZE,
    'cameras': ['left', 'center', 'right'],
    'aggregation': 'concat',
    'num_views': 3,
    'position_dim': 2,
    'force_torque_dim': 6,
    'joint_dim': 12,
    'target_mean': target_mean,
    'target_std': target_std,
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'saving to: {OUTPUT_DIR}')


In [ ]:
history = []
best_val = float('inf')

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    train_count = 0
    for batch in train_loader:
        batch = _move_batch(batch, DEVICE)
        optimizer.zero_grad(set_to_none=True)
        output = model(batch['image'], batch['force_torque'], batch['joint_positions'])
        loss = _loss(output, batch)
        loss.backward()
        optimizer.step()
        train_loss += float(loss.item()) * int(batch['image'].shape[0])
        train_count += int(batch['image'].shape[0])

    model.eval()
    val_loss = 0.0
    val_count = 0
    with torch.inference_mode():
        for batch in val_loader:
            batch = _move_batch(batch, DEVICE)
            output = model(batch['image'], batch['force_torque'], batch['joint_positions'])
            loss = _loss(output, batch)
            val_loss += float(loss.item()) * int(batch['image'].shape[0])
            val_count += int(batch['image'].shape[0])

    train_mean = train_loss / max(train_count, 1)
    val_mean = val_loss / max(val_count, 1)
    metrics = {'epoch': epoch, 'train_loss': train_mean, 'val_loss': val_mean}
    history.append(metrics)
    print(f"epoch={epoch:03d} train_loss={train_mean:.5f} val_loss={val_mean:.5f}")

    payload = {
        'model_state_dict': model.state_dict(),
        'config': config,
        'metrics': metrics,
    }
    torch.save(payload, OUTPUT_DIR / 'last.pt')
    if val_mean < best_val:
        best_val = val_mean
        torch.save(payload, OUTPUT_DIR / 'best.pt')
        print(f'  saved best.pt: val_loss={best_val:.5f}')

(OUTPUT_DIR / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')
print(f'done. best_val={best_val:.5f}')


In [ ]:
checkpoint = torch.load(OUTPUT_DIR / 'best.pt', map_location='cpu')
print(checkpoint['metrics'])
print(checkpoint['config']['versions'])
